<a href="https://colab.research.google.com/github/tejaswipriya22/Tejaswi_Ml-on-big-data/blob/main/Week6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [2]:
from sklearn.datasets import fetch_20newsgroups
from pyspark.sql import SparkSession
from pyspark.ml.feature import Tokenizer, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
import pandas as pd

# Start Spark Session
sprk = SparkSession.builder.appName("DocumentClassification").getOrCreate()

# Fetch 20 Newsgroups Data
newsgroups = fetch_20newsgroups(subset='all')

# Convert the dataset to a DataFrame for PySpark processing
data = pd.DataFrame({'text': newsgroups.data, 'category': newsgroups.target})
df = sprk.createDataFrame(data)

# Show some details about the data
print(f"Total number of documents: {len(newsgroups.data)}")
print(f"Categories: {newsgroups.target_names}")
print(f"Number of categories: {len(newsgroups.target_names)}")

# Display the distribution of categories
category_counts = df.groupBy('category').count().toPandas()
print("Category distribution before filtering (25%):")
print(category_counts)

# Filter 25% of documents from each category
df_sampled = df.sample(withReplacement=False, fraction=0.25, seed=42)

# Show the number of documents after sampling
total_documents_after_sampling = df_sampled.count()
print(f"Total number of documents after sampling: {total_documents_after_sampling}")

# Prepare for Document Classification

# Step 1: Tokenize the text
tokenizer = Tokenizer(inputCol="text", outputCol="words")

# Step 2: Apply HashingTF
hashingTF = HashingTF(inputCol="words", outputCol="raw_features", numFeatures=1000)

# Step 3: Compute IDF (Inverse Document Frequency)
idf = IDF(inputCol="raw_features", outputCol="features")

# Step 4: Convert category labels to numerical labels
indexer = StringIndexer(inputCol="category", outputCol="label")

# Step 5: Define the classifier (Logistic Regression in this case)
lr = LogisticRegression(featuresCol="features", labelCol="label")

# Set up the pipeline with all the stages
pipeline = Pipeline(stages=[tokenizer, hashingTF, idf, indexer, lr])

# Split the data into training and testing sets (80% train, 20% test)
train_data, test_data = df_sampled.randomSplit([0.8, 0.2], seed=42)

# Step 6: Train the model using the pipeline
model = pipeline.fit(train_data)

# Step 7: Make predictions on the test data
predictions = model.transform(test_data)

# Show some of the predictions
predictions.select("text", "category", "prediction").show(5, truncate=False)

# Step 8: Evaluate the model's accuracy
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)

# Display the accuracy
print(f"Model Accuracy: {accuracy:.2f}")

Total number of documents: 18846
Categories: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']
Number of categories: 20
Category distribution before filtering (25%):
    category  count
0         19    628
1          0    799
2          7    990
3          6    975
4          9    994
5         17    940
6          5    988
7          1    973
8         10    999
9          3    982
10        12    984
11         8    996
12        11    991
13         2    985
14         4    963
15        13    990
16        18    775
17        14    987
18        15    997
19        16    910
Total number of documents after sampling: 4773
+----------------------

In [3]:
!pip install pyspark nltk

In [4]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [5]:
# Import necessary PySpark modules
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

#Import NLP Libraries for Stemming & Lemmatization
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType
from nltk.stem import PorterStemmer, WordNetLemmatizer

In [6]:
#Define UDFs for Stemming & Lemmatization
# Initialize Stemmer and Lemmatizer
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Define UDF for Stemming
def stem_words(words):
    return [stemmer.stem(word) for word in words]

# Define UDF for Lemmatization
def lemmatize_words(words):
    return [lemmatizer.lemmatize(word) for word in words]

# Convert Python functions to PySpark UDFs
stem_udf = udf(stem_words, ArrayType(StringType()))
lemma_udf = udf(lemmatize_words, ArrayType(StringType()))

In [15]:
import pyspark
from pyspark.sql import SparkSession
import os

# Stop existing session
try:
    if 'spark' in globals() and spark is not None:
        spark.stop()
except:
    pass

# Create a new session with optimizations for Python UDFs and memory
spark = SparkSession.builder.appName("DocumentClassificationTFIDF") \
    .config("spark.driver.memory", "10g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

spark

In [8]:
from sklearn.datasets import fetch_20newsgroups
from pyspark.sql import SparkSession
from pyspark.ml.feature import Tokenizer, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
import pandas as pd

# Fetch 20 Newsgroups Data
newsgroups = fetch_20newsgroups(subset='all')

# Convert the dataset to a DataFrame for PySpark processing
data = pd.DataFrame({'text': newsgroups.data, 'category': newsgroups.target})
df = spark.createDataFrame(data)

# Show some details about the data
print(f"Total number of documents: {len(newsgroups.data)}")
print(f"Categories: {newsgroups.target_names}")
print(f"Number of categories: {len(newsgroups.target_names)}")

# Display the distribution of categories
category_counts = df.groupBy('category').count().toPandas()
print("Category distribution before filtering (25%):")
print(category_counts)

# Filter 25% of documents from each category
df_sampled = df.sample(withReplacement=False, fraction=0.25, seed=42)

# Show the number of documents after sampling
total_documents_after_sampling = df_sampled.count()
print(f"Total number of documents after sampling: {total_documents_after_sampling}")

Total number of documents: 18846
Categories: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']
Number of categories: 20
Category distribution before filtering (25%):
    category  count
0         19    628
1          0    799
2          7    990
3          6    975
4          9    994
5         17    940
6          5    988
7          1    973
8         10    999
9          3    982
10        12    984
11         8    996
12        11    991
13         2    985
14         4    963
15        13    990
16        18    775
17        14    987
18        15    997
19        16    910
Total number of documents after sampling: 4773


In [9]:
# Tokenization
tokenizer = Tokenizer(inputCol="text", outputCol="words")
df = tokenizer.transform(df)

# Stopword Removal
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
df = remover.transform(df)

# Apply Stemming
df = df.withColumn("stemmed_words", stem_udf(col("filtered_words")))

# Apply Lemmatization
df = df.withColumn("lemmatized_words", lemma_udf(col("filtered_words")))

# Show Results
df.select("text", "filtered_words", "stemmed_words", "lemmatized_words").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [10]:
# Compute TF-IDF After Preprocessing
# Apply HashingTF to the lemmatized words
hashingTF = HashingTF(inputCol="lemmatized_words", outputCol="raw_features", numFeatures=500)
df = hashingTF.transform(df)

# Compute IDF
idf = IDF(inputCol="raw_features", outputCol="features")
idf_model = idf.fit(df)
df = idf_model.transform(df)

# Show TF-IDF Features
df.select("text", "features").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [11]:
# Convert category labels to numerical labels
indexer = StringIndexer(inputCol="category", outputCol="label")
df = indexer.fit(df).transform(df)
df.select("category", "label").distinct().show()
# Split Data
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

# Train Logistic Regression Model
lr = LogisticRegression(featuresCol="features", labelCol="label")
lr_model = lr.fit(train_data)

# Predictions
predictions = lr_model.transform(test_data)
predictions.select("text", "category", "prediction").show(truncate=False)

# Evaluate Model Accuracy
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy_tf_idf = evaluator.evaluate(predictions)
print(f"TF-IDF Model Accuracy: {accuracy_tf_idf:.2f}")

+--------+-----+
|category|label|
+--------+-----+
|       2|  9.0|
|      15|  1.0|
|      19| 19.0|
|       3| 11.0|
|       8|  2.0|
|       0| 17.0|
|       6| 12.0|
|       7|  6.0|
|      16| 16.0|
|      14|  8.0|
|      13|  5.0|
|      17| 15.0|
|      11|  4.0|
|       1| 13.0|
|       4| 14.0|
|       5|  7.0|
|      18| 18.0|
|      12| 10.0|
|      10|  0.0|
|       9|  3.0|
+--------+-----+

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [12]:
df.select("category", "label").distinct().show()

+--------+-----+
|category|label|
+--------+-----+
|       2|  9.0|
|      15|  1.0|
|      19| 19.0|
|       3| 11.0|
|       8|  2.0|
|       0| 17.0|
|       6| 12.0|
|       7|  6.0|
|      16| 16.0|
|      14|  8.0|
|      13|  5.0|
|      17| 15.0|
|      11|  4.0|
|       1| 13.0|
|       4| 14.0|
|       5|  7.0|
|      18| 18.0|
|      12| 10.0|
|      10|  0.0|
|       9|  3.0|
+--------+-----+



In [21]:
from pyspark.ml.feature import Word2Vec, Tokenizer, StopWordsRemover
from pyspark.sql.functions import col
import pandas as pd

# 1. Re-create the base DataFrames using the NEW active Spark session
# We use the 'data' pandas DataFrame already in memory from cell FjW1uBi5hvI5
df_recreated = spark.createDataFrame(data)
df_sampled_recreated = df_recreated.sample(withReplacement=False, fraction=0.25, seed=42)

# 2. Basic Spark Transformations
tokenizer = Tokenizer(inputCol="text", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")

df_words = tokenizer.transform(df_sampled_recreated)
df_words = remover.transform(df_words)

# 3. Convert to Pandas to handle Lemmatization safely
pdf = df_words.select("text", "category", "filtered_words").toPandas()
pdf['lemmatized_words'] = pdf['filtered_words'].apply(lemmatize_words)

# 4. Convert back to Spark
df_sampled_proc = spark.createDataFrame(pdf)

# 5. Train Word2Vec
word2Vec = Word2Vec(vectorSize=100, minCount=2, inputCol="lemmatized_words", outputCol="featuresW2Vector")
word2Vec_model = word2Vec.fit(df_sampled_proc)
df_w2v = word2Vec_model.transform(df_sampled_proc)

# Show Results
df_w2v.select("text", "featuresW2Vector").show(5, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [24]:

# 1. Index the labels in the Word2Vec dataframe
indexer_w2v = StringIndexer(inputCol="category", outputCol="label")
df_w2v_final = indexer_w2v.fit(df_w2v).transform(df_w2v)

# 2. Split Data using the correct dataframe (df_w2v_final)
train_data_w2v, test_data_w2v = df_w2v_final.randomSplit([0.8, 0.2], seed=42)

# 3. Train Model
lr_w2v = LogisticRegression(featuresCol="featuresW2Vector", labelCol="label")
lr_w2v_model = lr_w2v.fit(train_data_w2v)

# 4. Predictions
predictions_w2v = lr_w2v_model.transform(test_data_w2v)
predictions_w2v.select("text", "category", "prediction").show(5, truncate=False)

# 5. Evaluate Model Accuracy
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy_w2v = evaluator.evaluate(predictions_w2v)
print(f"Word2Vec Model Accuracy: {accuracy_w2v:.2f}")

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [27]:
from pyspark.ml.classification import LinearSVC, OneVsRest

# 1. Use a much larger feature space (10,000 features) for TF-IDF
hashingTF_high = HashingTF(inputCol="lemmatized_words", outputCol="raw_features_high", numFeatures=10000)
df_high = hashingTF_high.transform(df_w2v_final)

idf_high = IDF(inputCol="raw_features_high", outputCol="features_high")
idf_model_high = idf_high.fit(df_high)
df_final_high = idf_model_high.transform(df_high)

# 2. Split Data
train_high, test_high = df_final_high.randomSplit([0.8, 0.2], seed=42)

# 3. Use Linear Support Vector Machine (SVC) with One-Vs-Rest for multiclass
lsvc = LinearSVC(maxIter=10, regParam=0.1, featuresCol="features_high", labelCol="label")
ovr = OneVsRest(classifier=lsvc, labelCol="label", featuresCol="features_high")

# Train the model
ovr_model = ovr.fit(train_high)

# 4. Predictions and Evaluation
predictions_high = ovr_model.transform(test_high)
accuracy_high = evaluator.evaluate(predictions_high)

print(f"High-Dimensional Linear SVC Accuracy: {accuracy_high:.2f}")

High-Dimensional Linear SVC Accuracy: 0.76
